# Question 24 - 68. Text Justification

Given an array of strings `words` and a width `maxWidth`, format the text such that each line has exactly `maxWidth` characters and is fully (left and right) justified.

You should pack your words in a greedy approach; that is, pack as many words as you can in each line. Pad extra spaces `' '` when necessary so that each line has exactly `maxWidth` characters.

Extra spaces between words should be distributed as evenly as possible. If the number of spaces on a line does not divide evenly between words, the empty slots on the left will be assigned more spaces than the slots on the right.

For the last line of text, it should be left-justified, and no extra space is inserted between words.

**Note:**

- A word is defined as a character sequence consisting of non-space characters only.
- Each word's length is guaranteed to be greater than `0` and not exceed `maxWidth`.
- The input array `words` contains at least one word.

**Example 1:**

    Input: words = ["This", "is", "an", "example", "of", "text", "justification."], maxWidth = 16
    Output:
    [
       "This    is    an",
       "example  of text",
       "justification.  "
    ]

**Example 2:**

    Input: words = ["What","must","be","acknowledgment","shall","be"], maxWidth = 16
    Output:
    [
      "What   must   be",
      "acknowledgment  ",
      "shall be        "
    ]
    Explanation: Note that the last line is "shall be    " instead of "shall     be", because the last line must be left-justified instead of fully-justified.
    Note that the second line is also left-justified because it contains only one word.

**Example 3:**

    Input: words = ["Science","is","what","we","understand","well","enough","to","explain","to","a","computer.","Art","is","everything","else","we","do"], maxWidth = 20
    Output:
    [
      "Science  is  what we",
      "understand      well",
      "enough to explain to",
      "a  computer.  Art is",
      "everything  else  we",
      "do                  "
    ]

**Constraints:**

- `1 <= words.length <= 300`
- `1 <= words[i].length <= 20`
- `words[i]` consists of only English letters and symbols.
- `1 <= maxWidth <= 100`
- `words[i].length <= maxWidth`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy line packing + arithmetic space distribution (divmod)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (round-robin one space at a time) | O(L · maxWidth) | O(maxWidth) |
| 2️⃣ Optimized (greedy pack + divmod) | O(S + L · maxWidth) | O(maxWidth) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"This is mostly about getting the rules exactly right, not about a clever algorithm. Let me list the rules so I don't miss one."*

1. **Packing is greedy:** put as many words on a line as fit, counting at least one space between words.
2. **Normal lines:** spread the extra spaces across the gaps. If they don't divide evenly, the **leftmost** gaps get one extra.
3. **A line with one word:** left-justify it and pad on the right (there are no gaps to fill).
4. **The last line:** left-justify with single spaces, and pad on the right.

- Every word fits on a line by itself (`len ≤ maxWidth`)? → **Yes.**
- Output: a list of strings, each exactly `maxWidth` long.

I'll structure the code as **two separate helpers**: "which words go on this line" and "how to lay out this line".

### Step 1 · Brute force

**Idea:** Pack lines greedily. For a normal line, start with one space in each gap, then hand out the remaining spaces **one at a time, round-robin from the left**, until the line is full.

🗣️ *"A natural first version distributes spaces one at a time, left to right. It's correct, and the left-gaps-get-more rule falls out naturally. But it loops once per extra space, and it hides the simple arithmetic. With divmod I can compute each gap's width directly."*

In [1]:
class SolutionBrute:
    def fullJustify(self, words: list[str], maxWidth: int) -> list[str]:
        lines, i = [], 0
        while i < len(words):
            # greedy pack
            j, width = i, 0
            while j < len(words) and width + len(words[j]) + (j - i) <= maxWidth:
                width += len(words[j])
                j += 1
            line_words = words[i:j]
            gaps = len(line_words) - 1
            if j == len(words) or gaps == 0:
                line = " ".join(line_words)
                lines.append(line + " " * (maxWidth - len(line)))
            else:
                spaces = [1] * gaps
                remaining = maxWidth - width - gaps
                k = 0
                while remaining > 0:            # one space at a time, round-robin
                    spaces[k % gaps] += 1
                    remaining -= 1
                    k += 1
                line = line_words[0]
                for g, w in zip(spaces, line_words[1:]):
                    line += " " * g + w
                lines.append(line)
            i = j
        return lines

### Step 2 · Optimize

**Bottleneck:** handing out spaces one at a time. The real challenge in this problem is correctness, not speed, so the optimised version aims to be **simpler and direct**.

**Key insight: the distribution is just integer division.**

For a line with words of total length `width` and `gaps = k − 1` gaps:
- `total_spaces = maxWidth − width`
- `base, extra = divmod(total_spaces, gaps)`
- Gaps `0 … extra−1` get `base + 1` spaces; the rest get `base`.

**Packing condition:** adding word `j` keeps the line valid if `width + len(words[j]) + (number of words already on the line) ≤ maxWidth`, because each existing word needs at least one space before the new one.

🗣️ *"I loop over lines. For each line I greedily take words while the total letters plus one space per gap fits. Then, if it's the last line or has a single word, I left-justify and pad. Otherwise I divmod the spare spaces by the number of gaps: everyone gets the base amount and the first `extra` gaps get one more. Each character is written once, so it's linear in the output size."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Round-robin one space at a time | ⚠️ | Correct, but loops per space and hides the rule. |
| **Greedy packing + `divmod`** | ✅ | The problem *requires* greedy packing, and divmod encodes "spread evenly, leftmost get extra" exactly. |
| DP to minimise raggedness (Knuth–Plass) | ❌ | That's a **different** problem (pretty-printing). This problem mandates greedy, so DP would give a different answer. |
| Build lines with `str +=` | ⚠️ | Fine at this size, but `list` + `"".join` is the linear-time habit to show. |
| `str.ljust(maxWidth)` for padding | ✅ | Clean way to pad the last line and single-word lines. |

In [2]:
class Solution:
    def fullJustify(self, words: list[str], maxWidth: int) -> list[str]:
        result = []
        i, n = 0, len(words)
        while i < n:
            # 1) greedy pack words[i:j] onto this line
            j, letters = i, 0
            while j < n and letters + len(words[j]) + (j - i) <= maxWidth:
                letters += len(words[j])
                j += 1
            line_words = words[i:j]
            gaps = j - i - 1

            # 2) lay out the line
            if j == n or gaps == 0:
                # last line or single word: left-justify, pad right
                result.append(" ".join(line_words).ljust(maxWidth))
            else:
                base, extra = divmod(maxWidth - letters, gaps)
                parts = []
                for k, w in enumerate(line_words[:-1]):
                    parts.append(w)
                    parts.append(" " * (base + (1 if k < extra else 0)))
                parts.append(line_words[-1])
                result.append("".join(parts))
            i = j
        return result

### Step 3 · Toy example walkthrough

`words = ["This", "is", "an", "example", "of", "text", "justification."]`, `maxWidth = 16`

**Line 1, packing:**

| try word | letters so far | + len + gaps-needed | ≤ 16? |
|---|---|---|---|
| This (4) | 0 | 0+4+0 = 4 | ✅ |
| is (2) | 4 | 4+2+1 = 7 | ✅ |
| an (2) | 6 | 6+2+2 = 10 | ✅ |
| example (7) | 8 | 8+7+3 = 18 | ❌ stop |

Line words = `This is an`, letters = 8, gaps = 2.
`divmod(16 − 8, 2)` = (4, 0) → every gap gets 4 → `"This····is····an"` ✅

**Line 2:** `example of text` → letters = 13, gaps = 2.
`divmod(16 − 13, 2)` = (1, **1**) → gap 0 gets **2**, gap 1 gets 1 → `"example··of·text"` ✅ (the left gap gets the extra space)

**Line 3:** `justification.`. This is the last line → left-justify → `"justification.··"` ✅

In [3]:
cases = [
    (["This", "is", "an", "example", "of", "text", "justification."], 16,
     ["This    is    an", "example  of text", "justification.  "]),
    (["What", "must", "be", "acknowledgment", "shall", "be"], 16,
     ["What   must   be", "acknowledgment  ", "shall be        "]),
    (["Science", "is", "what", "we", "understand", "well", "enough", "to", "explain",
      "to", "a", "computer.", "Art", "is", "everything", "else", "we", "do"], 20,
     ["Science  is  what we", "understand      well", "enough to explain to",
      "a  computer.  Art is", "everything  else  we", "do                  "]),
    (["a"], 1, ["a"]),
    (["a", "b", "c"], 1, ["a", "b", "c"]),
]
for words, w, expected in cases:
    assert SolutionBrute().fullJustify(words, w) == expected
    got = Solution().fullJustify(words, w)
    assert got == expected
    assert all(len(line) == w for line in got)
    for line in got:
        print(f"|{line}|")
    print()
print("All tests passed ✅")

|This    is    an|
|example  of text|
|justification.  |

|What   must   be|
|acknowledgment  |
|shall be        |

|Science  is  what we|
|understand      well|
|enough to explain to|
|a  computer.  Art is|
|everything  else  we|
|do                  |

|a|

|a|
|b|
|c|

All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Round-robin spaces | O(L · maxWidth) | O(maxWidth) | Loops per extra space on each of the L lines |
| **Greedy + divmod** | **O(S + L · maxWidth)** | **O(maxWidth)** extra | Each word is examined once while packing; each output character is written once |

`S` = total characters in `words`, `L` = number of output lines.

**Complexity, explained:** the packing loop visits each word exactly once overall (`i` jumps to `j`). Building each line writes exactly `maxWidth` characters. So the total work is the input size plus the output size. Apart from the output, we only hold one line's parts at a time → O(maxWidth).

**Rules checklist (say these out loud, they're where people fail):**
1. Pack condition counts **one space per existing word** → `letters + len(w) + (j − i)`.
2. **Last line** → single spaces, pad right.
3. **Single-word line** → pad right (avoid dividing by zero gaps!).
4. Uneven spaces → **leftmost** gaps get the extra.

**Edge cases:** `maxWidth` equal to a word length, every word on its own line, a single word overall.

🗣️ **30-second recap:** *"It's a simulation with two steps per line. First, greedily pack words while letters plus one space per gap fits. Second, lay out the line: if it's the last line or has one word, join with single spaces and pad right; otherwise divmod the spare spaces by the number of gaps, give each gap the base, and give the first `extra` gaps one more. Linear in input plus output."*